In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global seaborn / matplotlib defaults
sns.set_theme(
    style="whitegrid",  # axes with grid
    rc={
        "grid.linestyle": "-",
        "grid.alpha": 0.6,
    },
)

## Load data

In [ ]:
import pandas as pd


def preview_results(df: pd.DataFrame, sample_size: int = 10) -> None:
    if len(df) > 0:
        display(df.sample(min(sample_size, len(df))))


logprob_results_path = "../analysis/results/logprob_acc_merged.json"
logprob_results = pd.read_json(logprob_results_path, orient="records")

logprob_norm_results_path = "../analysis/results/logprob_acc_norm_merged.json"
logprob_norm_results = pd.read_json(logprob_norm_results_path, orient="records")

generative_results_path = "../analysis/generative_tails.json"
generative_results = pd.read_json(generative_results_path, orient="records")

In [ ]:
# to extract the L2 norm and other non metric values, we need to load the raw results

raw_results_path = "../logs/silent-norm-final-v1/results.json"
raw_results = pd.read_json(raw_results_path, orient="records")

In [ ]:
def add_path_metadata(dirty_df: pd.DataFrame) -> pd.DataFrame:
    dirty_df = dirty_df.copy()
    dirty_parts = dirty_df["path"].str.split("/")
    if (dirty_parts.str.len() < 2).any():
        raise ValueError("Clean path does not contain enough segments to parse model_name")

    dirty_out = dirty_df.copy()

    dirty_out["model_name"] = dirty_parts.str[-5]
    dirty_out["train_dataset"] = dirty_parts.str[-4]
    dirty_out["layer_name"] = dirty_parts.str[-3]
    dirty_out["exp_name"] = dirty_parts.str[-2]

    return dirty_out


# apply formatting to metric column
def format_metric_column(df: pd.DataFrame, metric_col: str = "metric") -> pd.DataFrame:
    df = df.copy()
    df[metric_col] = df[metric_col].apply(lambda x: x.replace(",none", ""))
    return df


raw_results = add_path_metadata(raw_results)
raw_results = format_metric_column(raw_results)


raw_results.sample(3)

In [ ]:
def extract_kl_val(row: pd.Series) -> float:
    # Llama-2-7b-chat-hf-KL-0.0-iter1
    exp_name = row["exp_name"]
    kl_str = exp_name.split("KL-")[-1].split("-")[0]
    return float(kl_str)


logprob_results["kl"] = logprob_results.apply(extract_kl_val, axis=1)
logprob_norm_results["kl"] = logprob_norm_results.apply(extract_kl_val, axis=1)
generative_results["kl"] = generative_results.apply(extract_kl_val, axis=1)
raw_results["kl"] = raw_results.apply(extract_kl_val, axis=1)

## Process Raw Results

In [ ]:
SINGLE_EXPERIMENT_KEYS = ["model_name", "layer_name", "kl"]

In [ ]:
raw_results["benchmark_metric"] = raw_results["benchmark"] + "/" + raw_results["metric"]
pivot_raw_results = raw_results.pivot_table(index=SINGLE_EXPERIMENT_KEYS, columns="benchmark_metric", values="value")

In [ ]:
denom = 15 + 50


# Compute global metrics as weighted averages of the two benchmarks
pivot_raw_results["global/kl_div"] = (15 / denom) * pivot_raw_results["eval-oasst2/kl_div"] + (50 / denom) * pivot_raw_results["eval-tulu-v3/kl_div"]
pivot_raw_results["global/proj_l2_rel"] = (15 / denom) * pivot_raw_results["eval-oasst2/proj_l2_rel"] + (50 / denom) * pivot_raw_results[
    "eval-tulu-v3/proj_l2_rel"
]


In [ ]:
# Get the reference values where kl == 0
ref_values = pivot_raw_results.xs(0.0, level="kl")[["global/kl_div", "global/proj_l2_rel"]]

# Join reference values to the pivot table
pivot_raw_results = pivot_raw_results.join(ref_values, on=["model_name", "layer_name"], rsuffix="_max")
pivot_raw_results = pivot_raw_results[["global/kl_div", "global/kl_div_max", "global/proj_l2_rel", "global/proj_l2_rel_max"]]

### Concat Results

In [ ]:
# add unique identifiers to the metric results for merging later
logprob_results["metric"] = "acc"
logprob_norm_results["metric"] = "acc_norm"
generative_results["metric"] = "generative"

In [ ]:
# merge logprob results with logprob norm results
logprob_results = pd.concat([logprob_results, logprob_norm_results], ignore_index=True)
logprob_results

In [ ]:
# merge with raw results to get the kl_div and proj_l2_rel values for each experiment

logprob_results = logprob_results.merge(
    pivot_raw_results.reset_index(),
    on=SINGLE_EXPERIMENT_KEYS,
    how="left",
)

generative_results = generative_results.merge(
    pivot_raw_results.reset_index(),
    on=SINGLE_EXPERIMENT_KEYS,
    how="left",
)

# now we have all the info in logprob_results and generative_results, we can start analyzing the results

In [ ]:
def abs_diff(df: pd.DataFrame, col1: str, col2: str, new_col: str = "abs_diff") -> pd.DataFrame:
    df[new_col] = (df[col1] - df[col2]).abs()
    return df


# add absolute difference between dirty_mean and clean_mean
logprob_results = abs_diff(logprob_results, "dirty_mean", "clean_mean", new_col="abs_diff")
generative_results = abs_diff(generative_results, "value", "clean_mean", new_col="abs_diff")

In [ ]:
# compute silentness score for each benchmark and metric
SILENCE_SCORE_COL = "silence_score"

logprob_results[SILENCE_SCORE_COL] = logprob_results["two_sided_tail"] + logprob_results["diff_prob"]
generative_results[SILENCE_SCORE_COL] = generative_results["two_sided_tail"] + generative_results["diff_prob"]

## Now Aggregate Silentness Score per Experiment

In [ ]:
def agg_results(
    df: pd.DataFrame,
    prob_col: str = "two_sided_tail",
    group_by_cols: list[str] | None = None,
    keep_cols: list[str] | None = None,
) -> pd.DataFrame:
    keep_cols_local = list(keep_cols) if keep_cols is not None else []
    keep_cols_local = list(dict.fromkeys([*keep_cols_local, prob_col]))  # unique, order-preserving

    group_cols = list(group_by_cols) if group_by_cols is not None else []
    selected_cols = list(dict.fromkeys([*group_cols, *keep_cols_local]))

    if group_by_cols is not None:
        idx = df.groupby(group_by_cols, sort=False)[prob_col].idxmin()
        agg_df = df.loc[idx, selected_cols].reset_index(drop=True)
    else:
        idx = df[prob_col].idxmin()
        agg_df = df.loc[[idx], selected_cols].reset_index(drop=True)

    return agg_df


raw_keep_cols = [
    "global/kl_div",
    "global/kl_div_max",
    "global/proj_l2_rel",
    "global/proj_l2_rel_max",
    "diff_prob",
    "two_sided_tail",
    "lower_tail",
    "upper_tail",
    "benchmark",
]

agg_logprobs = agg_results(
    logprob_results.copy(),
    prob_col=SILENCE_SCORE_COL,
    group_by_cols=SINGLE_EXPERIMENT_KEYS,  # experiment level aggregation
    keep_cols=raw_keep_cols,
)

agg_generative = agg_results(
    generative_results.copy(),
    prob_col=SILENCE_SCORE_COL,
    group_by_cols=SINGLE_EXPERIMENT_KEYS,  # experiment level aggregation
    keep_cols=raw_keep_cols,
)

In [ ]:
KL_CHOICE_COL = "kl_metric"
# given same experiment over many KL values, we need to choose a single KL value to represent the experiment. This column will be used for that choice, and will be computed as the product of the silence score and the global/proj_l2_rel metric, which captures both the silence and the distance from the reference direction.
# this metric takes into account both the silence score and also the energy of the projection, aiming to maximize both.

agg_logprobs[KL_CHOICE_COL] = agg_logprobs["global/proj_l2_rel"] * agg_logprobs[SILENCE_SCORE_COL]
agg_generative[KL_CHOICE_COL] = agg_generative["global/proj_l2_rel"] * agg_generative[SILENCE_SCORE_COL]

In [ ]:
agg_choice = agg_logprobs.merge(
    agg_generative,
    on=SINGLE_EXPERIMENT_KEYS,
    suffixes=("_logprob", "_generative"),
)

# the global/ metrics are the same for both logprob and generative results, so lets remove the redundant generative ones
# and rename the logprob ones to be without the suffix

agg_choice = agg_choice.drop(
    columns=[
        "global/kl_div_generative",
        "global/kl_div_max_generative",
        "global/proj_l2_rel_generative",
        "global/proj_l2_rel_max_generative",
    ]
)

agg_choice = agg_choice.rename(
    columns={
        "global/kl_div_logprob": "global/kl_div",
        "global/kl_div_max_logprob": "global/kl_div_max",
        "global/proj_l2_rel_logprob": "global/proj_l2_rel",
        "global/proj_l2_rel_max_logprob": "global/proj_l2_rel_max",
    }
)

In [ ]:
# aggregate the kl choice and silence score by taking the min of the logprob and generative versions
agg_choice[KL_CHOICE_COL] = agg_choice[[f"{KL_CHOICE_COL}_logprob", f"{KL_CHOICE_COL}_generative"]].min(axis=1)
agg_choice[SILENCE_SCORE_COL] = agg_choice[[f"{SILENCE_SCORE_COL}_logprob", f"{SILENCE_SCORE_COL}_generative"]].min(axis=1)

In [ ]:
# Cleaner: break ties with kl first, then pick max agg_choice_metric per group via idxmax.
_tmp = agg_choice.sort_values(SINGLE_EXPERIMENT_KEYS, ascending=[True, True, False])
_idx = _tmp.groupby(["model_name", "layer_name"])[KL_CHOICE_COL].idxmax()

best_kl_df = (_tmp.loc[_idx, agg_choice.columns.tolist()].sort_values(["model_name", "layer_name"])).reset_index(drop=True)

best_kl_df

In [ ]:
rename_map = {
    # Experiment
    "model_name": "Model",
    "layer_name": "Layer",
    "kl": r"$\lambda$",
    # Experiment Results
    SILENCE_SCORE_COL: r"$\mathrm{Sil}$",
    "global/proj_l2_rel": r"$E_{L2}$",
    "global/proj_l2_rel_max": r"$\max E_{L2}$",
    "global/kl_div": r"$\mathcal{L}_{KL}$",
    "global/kl_div_max": r"$\max \mathcal{L}_{KL}$",
    # Logprob Results
    "diff_prob_logprob": r"$\mathrm{Diff}_{\text{logprob}}$",
    "two_sided_tail_logprob": r"$\mathrm{Tail}_{\text{logprob}}$",
    f"{SILENCE_SCORE_COL}_logprob": r"$\mathrm{Sil}_{\text{logprob}}$",
    # Generative Results
    "diff_prob_generative": r"$\mathrm{Diff}_{\text{generative}}$",
    "two_sided_tail_generative": r"$\mathrm{Tail}_{\text{generative}}$",
    f"{SILENCE_SCORE_COL}_generative": r"$\mathrm{Sil}_{\text{generative}}$",
    "benchmark_generative": "benchmark_generative",
    "benchmark_logprob": "benchmark_logprob",
}
mask_cols = [k for k in rename_map.keys()]
best_kl_df_renamed = best_kl_df[mask_cols].rename(columns=rename_map)

In [ ]:
best_kl_df_renamed.sort_values(rename_map[SILENCE_SCORE_COL], ascending=False)

In [ ]:
best_kl_df_renamed

In [ ]:
def fmt(x, k=4):
    return f"{x:.{k}f}".rstrip("0").rstrip(".")


print(best_kl_df_renamed.to_latex(index=False, float_format=lambda x: fmt(x, k=4)))

In [ ]:
def layer_order(layer_name: str) -> int:
    if ".layers." in layer_name:
        return int(layer_name.split(".")[-1])
    elif "embed_tokens" in layer_name:
        return -1
    else:
        return 1000

In [ ]:
def visualize(
    df: pd.DataFrame,
    res_name: str,
    abs_diff_threshold: float,
    row: str = None,
):
    g = sns.catplot(
        data=df,
        x="two_sided_tail_agg",
        y="layer_name",
        kind="bar",
        col="model_name",
        row=row,
        order=sorted(df["layer_name"].unique(), key=layer_order),
    )
    g.figure.suptitle(f"Aggregated {res_name} two-sided tail values (threshold={abs_diff_threshold})", y=1.02)